# Lifelong Learning Lab: Domain-Incremental Deception Detection with Replay

Companion notebook for *AI for Security and Security for AI*.


## Goal

This notebook demonstrates continual learning over a sequence of security/deception domains. We use the seven DiFrauD domains as a domain-incremental stream.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
DIFRAUD_BASE = "https://huggingface.co/datasets/difraud/difraud/resolve/main"

def load_difraud_domain(domain, splits=("train", "validation", "test")):
    frames = []
    for split in splits:
        url = f"{DIFRAUD_BASE}/{domain}/{split}.jsonl"
        tmp = pd.read_json(url, lines=True)
        tmp["split"] = split
        tmp["domain"] = domain
        frames.append(tmp)
    df = pd.concat(frames, ignore_index=True)
    df = df.dropna(subset=["text", "label"])
    df["text"] = df["text"].astype(str)
    df["label"] = df["label"].astype(int)
    return df

DOMAINS = ["phishing", "fake_news", "political_statements", "product_reviews", "job_scams", "sms", "twitter_rumours"]
all_domains = {}
for d in DOMAINS:
    print("Loading", d)
    all_domains[d] = load_difraud_domain(d)
summary = []
for d, frame in all_domains.items():
    for split, g in frame.groupby("split"):
        summary.append({"domain": d, "split": split, "n": len(g), "positive_rate": g["label"].mean()})
pd.DataFrame(summary).pivot(index="domain", columns="split", values="n")


## Experiment design

Each DiFrauD domain is treated as a task. After training on each task, the model is evaluated on all tasks seen so far.


In [ ]:
from collections import defaultdict
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

MAX_TRAIN_PER_DOMAIN = 5000
MAX_TEST_PER_DOMAIN = 1500
CLASSES = np.array([0, 1])
vectorizer = HashingVectorizer(n_features=2**18, alternate_sign=False, ngram_range=(1, 2), norm="l2")

class ClassBalancedReplayBuffer:
    def __init__(self, capacity=1000):
        self.capacity = capacity
        self.store = defaultdict(list)
    def add_batch(self, texts, labels):
        for x, y in zip(texts, labels):
            self.store[int(y)].append((str(x), int(y)))
        self._trim()
    def _trim(self):
        active = list(self.store.keys())
        if not active: return
        per_class = max(1, self.capacity // len(active))
        for c in active:
            if len(self.store[c]) > per_class:
                self.store[c] = self.store[c][-per_class:]
    def sample(self, n, random_state=RANDOM_STATE):
        rng = np.random.default_rng(random_state)
        items = []
        active = list(self.store.keys())
        if not active: return [], np.array([], dtype=int)
        per_class = max(1, n // len(active))
        for c in active:
            pool = self.store[c]
            if not pool: continue
            idx = rng.choice(len(pool), size=min(per_class, len(pool)), replace=False)
            items.extend([pool[i] for i in idx])
        if not items: return [], np.array([], dtype=int)
        texts, labels = zip(*items)
        return list(texts), np.array(labels, dtype=int)

def sample_split(frame, split, max_n, random_state=RANDOM_STATE):
    out = frame[frame["split"] == split].copy()
    if len(out) > max_n:
        out = out.groupby("label", group_keys=False).apply(
            lambda g: g.sample(n=min(len(g), max(1, int(max_n * len(g) / len(out)))), random_state=random_state)
        )
        if len(out) > max_n:
            out = out.sample(max_n, random_state=random_state)
    return out.sample(frac=1, random_state=random_state).reset_index(drop=True)

def evaluate(clf, texts, labels):
    X = vectorizer.transform(texts)
    pred = clf.predict(X)
    p, r, f1, _ = precision_recall_fscore_support(labels, pred, average="binary", zero_division=0)
    return {"accuracy": accuracy_score(labels, pred), "precision": p, "recall": r, "f1": f1}

def average_forgetting(acc_matrix):
    final = acc_matrix.iloc[-1]
    best = acc_matrix.max(axis=0)
    return float((best - final).dropna().mean())


In [ ]:
def run_continual_experiment(use_replay=False, memory_size=1000, replay_ratio=0.5):
    clf = SGDClassifier(loss="log_loss", alpha=1e-5, random_state=RANDOM_STATE)
    initialized = False
    buffer = ClassBalancedReplayBuffer(capacity=memory_size) if use_replay else None
    test_sets = {}
    acc_rows = []
    f1_rows = []
    for step, domain in enumerate(DOMAINS):
        frame = all_domains[domain]
        train_df = sample_split(frame, "train", MAX_TRAIN_PER_DOMAIN)
        test_df = sample_split(frame, "test", MAX_TEST_PER_DOMAIN)
        train_texts = list(train_df["text"])
        train_labels = train_df["label"].values
        if use_replay:
            n_replay = int(replay_ratio * len(train_texts))
            replay_texts, replay_labels = buffer.sample(n_replay, random_state=RANDOM_STATE + step)
            train_texts = train_texts + replay_texts
            train_labels = np.concatenate([train_labels, replay_labels])
        X_train = vectorizer.transform(train_texts)
        if not initialized:
            clf.partial_fit(X_train, train_labels, classes=CLASSES)
            initialized = True
        else:
            clf.partial_fit(X_train, train_labels)
        if use_replay:
            buffer.add_batch(list(train_df["text"]), train_df["label"].values)
        test_sets[domain] = test_df
        acc_row = {}
        f1_row = {}
        for eval_domain, eval_df in test_sets.items():
            m = evaluate(clf, list(eval_df["text"]), eval_df["label"].values)
            acc_row[eval_domain] = m["accuracy"]
            f1_row[eval_domain] = m["f1"]
        acc_rows.append(acc_row)
        f1_rows.append(f1_row)
    acc_matrix = pd.DataFrame(acc_rows, index=DOMAINS)
    f1_matrix = pd.DataFrame(f1_rows, index=DOMAINS)
    return clf, acc_matrix, f1_matrix, average_forgetting(acc_matrix)

naive_clf, naive_acc, naive_f1, naive_forgetting = run_continual_experiment(use_replay=False)
replay_clf, replay_acc, replay_f1, replay_forgetting = run_continual_experiment(use_replay=True, memory_size=1000, replay_ratio=0.5)
print("Naive average forgetting:", round(naive_forgetting, 4))
display(naive_acc.round(3))
print("Replay average forgetting:", round(replay_forgetting, 4))
display(replay_acc.round(3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
im0 = axes[0].imshow(naive_acc.fillna(0).values, vmin=0, vmax=1)
axes[0].set_title("Naive sequential training")
axes[0].set_xticks(range(len(naive_acc.columns)))
axes[0].set_xticklabels(naive_acc.columns, rotation=90)
axes[0].set_yticks(range(len(naive_acc.index)))
axes[0].set_yticklabels(naive_acc.index)
im1 = axes[1].imshow(replay_acc.fillna(0).values, vmin=0, vmax=1)
axes[1].set_title("Class-balanced replay")
axes[1].set_xticks(range(len(replay_acc.columns)))
axes[1].set_xticklabels(replay_acc.columns, rotation=90)
axes[1].set_yticks(range(len(replay_acc.index)))
axes[1].set_yticklabels(replay_acc.index)
fig.colorbar(im1, ax=axes.ravel().tolist(), shrink=0.85, label="accuracy")
plt.show()


## Practitioner takeaway

In the accuracy matrix, each row is the model after training on a new domain and each column is evaluation on a domain. If earlier columns decline as later domains are learned, the model is forgetting.
